# ACF portable training (Colab)

Rules of this notebook:

- It contains **no upload code** and mounts nothing. Where the data and the outputs live is your decision: fill `DATA_ROOT` and `OUT_ROOT` yourself.
- It **refuses real data** unless `ALLOW_REAL_DATA=1`. Set that only after written permission from the data provider to place the data on this machine.
- Cell 3 runs the whole loop on random tensors (`--synthetic`): no data, no cache. Use it for the kill-and-resume check.
- The test set is never loaded by `scripts/train.py`.

In [ ]:
# 1. Fill these two. Leave DATA_ROOT empty for the synthetic check.
DATA_ROOT = ""   # folder that holds <case>.acf and pet_scale.json (real data: only with permission)
OUT_ROOT = ""    # folder for runs/<name>/ (checkpoints, logs, manifest); must survive a session restart
CONFIG = "configs/shared_b4_acf_cond.yaml"
BRANCH = "fable/first-run"
assert OUT_ROOT, "fill OUT_ROOT"

In [ ]:
# 2. Code only: clone the repository (no data is in it) and install the requirements.
import os, subprocess
if not os.path.isdir("Hecktor"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/SUDER14/Hecktor.git"], check=True)
os.chdir("Hecktor")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# 3. Synthetic kill-and-resume check (no data). Run this cell, interrupt it (Runtime > Interrupt) after the first
#    "val subset" line, then run it again: it must print "resumed from step_..." and finish at step 60.
import subprocess, sys
subprocess.run([sys.executable, "scripts/train.py", "--config", CONFIG, "--out-root", OUT_ROOT, "--synthetic",
                "--resume", "auto", "--set", "steps.max_steps=60", "--set", "steps.val_every=20",
                "--set", "steps.val_subset_cases=2", "--set", "steps.ckpt_every_minutes=1"], check=True)

In [ ]:
# 4. Real training. Refuses to start unless ALLOW_REAL_DATA=1 (set it yourself, in a cell you write, only after
#    written permission from the data provider). Re-run this same cell after a session restart: it resumes.
import os, subprocess, sys
assert os.environ.get("ALLOW_REAL_DATA") == "1", "real data not allowed: ALLOW_REAL_DATA is not 1"
assert DATA_ROOT and os.path.isdir(DATA_ROOT), "fill DATA_ROOT"
subprocess.run([sys.executable, "scripts/train.py", "--config", CONFIG, "--data-root", DATA_ROOT,
                "--out-root", OUT_ROOT, "--resume", "auto"], check=True)

In [ ]:
# 5. Where the run stands.
import json, glob
for m in glob.glob(os.path.join(OUT_ROOT, "runs", "*", "manifest.json")):
    d = json.load(open(m)); print(d["name"], "step", d["step"], "/", d["max_steps"], "best", d["best_score"], "finished", d["finished"], "gpu_hours", d["gpu_hours"])